# TT-17 — Random Forest Regressor: Dự đoán giá vé máy bay để tư vấn "nên mua bây giờ hay chờ"

**Bài toán:** một đại lý vé máy bay trực tuyến (OTA) muốn hiển thị cho khách: *"Giá hiện tại
X — dự báo tuần sau Y → NÊN MUA NGAY hay CHỜ THÊM"*. Giá vé phụ thuộc phi tuyến vào nhiều yếu
tố có tương tác chéo (số ngày còn lại tới chuyến bay, hãng bay, hạng vé, số điểm dừng, giờ khởi
hành...) — đây là kịch bản mà **ensemble cây** (Random Forest) phù hợp hơn hẳn model tuyến tính.

**Dữ liệu:** [Flight Price Prediction (Kaggle)](https://www.kaggle.com/datasets/shubhambathwal/flight-price-prediction)
— ~300.000 chuyến bay nội địa Ấn Độ (2022), nhãn `price` (Rupee).

> ⚠️ Bộ dữ liệu gốc nằm trên Kaggle và cần tài khoản + API token để tải bằng `kaggle datasets
> download`. Vì môi trường chạy notebook này không có sẵn thông tin đăng nhập Kaggle của bạn,
> file `data/Clean_Dataset.csv` được lấy từ một **bản mirror công khai, giữ nguyên nội dung**
> của đúng bộ dữ liệu này trên GitHub (dự án học thuật `Amanrathi-Git/Flight-Pricing-Analytics`,
> cùng chủ đề Random Forest cho giá vé). Đã kiểm tra: **đúng 300.153 dòng**, đúng 11 cột mô tả
> trong README (`airline, flight, source_city, departure_time, stops, arrival_time,
> destination_city, class, duration, days_left, price`) — dữ liệu thật, không phải dữ liệu giả
> lập. Nếu bạn có tài khoản Kaggle, chỉ cần tải `Clean_Dataset.csv` trực tiếp và ghi đè vào
> `data/` — toàn bộ code phía dưới không cần đổi gì.

## Random Forest Regressor là gì

Huấn luyện nhiều **cây hồi quy** (TT-16) độc lập, mỗi cây học trên một mẫu **bootstrap** khác
nhau (lấy lại có hoàn lại từ tập train) và chỉ được xét một tập con đặc trưng ngẫu nhiên tại
mỗi lần chia nhánh. Kết quả dự đoán cuối cùng = **TRUNG BÌNH CỘNG** dự đoán của tất cả các cây
(khác với bản phân loại — ở đó là **bỏ phiếu đa số**).

```
   Cay 1 -> 3,2 trieu -+
   Cay 2 -> 3,5 trieu -+-> TRUNG BINH = 3,37 trieu
   Cay 3 -> 3,4 trieu -+

   Loi ich kep:
     (1) Ham du doan MUOT hon cay don (khong con bac thang tho)
     (2) On dinh hon - doi chut du lieu train khong lam ket qua nhay vot
```

⚠️ **Điểm yếu vẫn giữ nguyên từ cây đơn: KHÔNG NGOẠI SUY ĐƯỢC.** Mỗi cây chỉ có thể trả về
trung bình của một trong các lá đã học từ dữ liệu train — giá vé ứng với `days_left` vượt ngoài
khoảng đã thấy trong tập train (ở đây tối đa 49 ngày) sẽ bị "kẹp trần" ở mức giá của lá cuối
cùng, xem thí nghiệm ở bước 11.

In [ ]:
%matplotlib inline
import json
import time
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import partial_dependence, permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

RANDOM_STATE = 42
DATA_DIR = Path("../data")
REPORTS_DIR = Path("../reports")
MODELS_DIR = Path("../models")
for d in (REPORTS_DIR, MODELS_DIR):
    d.mkdir(exist_ok=True, parents=True)

CAT_COLS = ["airline", "source_city", "departure_time", "stops", "arrival_time", "destination_city", "class"]
NUM_COLS = ["duration", "days_left"]
FEATURES = CAT_COLS + NUM_COLS
TARGET = "price"


def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

## 1. Nạp dữ liệu, bỏ cột `flight` và cột chỉ số thừa

`flight` là mã hiệu chuyến bay (ví dụ `SG-8709`) — có **hàng nghìn giá trị duy nhất**. Nếu
đưa thẳng vào `OneHotEncoder`, nó sẽ tạo ra hàng nghìn cột nhị phân, và cây/rừng sẽ "học thuộc"
từng mã chuyến cụ thể thay vì học quy luật giá tổng quát (ví dụ: một mã chuyến chỉ xuất hiện vài
lần trong tập train sẽ khiến model ghi nhớ đúng giá của riêng nó — không khái quát hoá được sang
chuyến bay mới). Cột đầu tiên (`Unnamed: 0`) chỉ là chỉ số dòng khi file CSV được xuất ra, không
mang thông tin — cũng loại bỏ.

In [ ]:
df_raw = pd.read_csv(DATA_DIR / "Clean_Dataset.csv")
print(f"Du lieu goc: {df_raw.shape[0]:,} dong x {df_raw.shape[1]} cot")

df = df_raw.drop(columns=[c for c in df_raw.columns if c.startswith("Unnamed")])
df = df.drop(columns=["flight"])
print(f"Sau khi bo 'flight' va cot chi so: {df.shape[0]:,} dong x {df.shape[1]} cot")
print(f"\nSo gia tri thieu tren tung cot:\n{df.isna().sum()}")
df.head()

**Đọc kết quả:** dữ liệu gốc có đúng **300.153 dòng × 12 cột** — khớp README ("~300.000
dòng × 11 cột", chênh 1 cột do có thêm cột chỉ số `Unnamed: 0` khi xuất CSV). Sau khi bỏ
`Unnamed: 0` và `flight`, còn lại **300.153 dòng × 10 cột** (9 đặc trưng + 1 nhãn `price`).
**Không có giá trị thiếu ở bất kỳ cột nào** — hiếm gặp với dữ liệu thực tế cỡ này, nghĩa là
không cần bước xử lý missing value (khác với TT-16, nơi `passenger_count` thiếu tới ~7.000
dòng). Các biến phân loại có số mức: `airline` (6), `source_city`/`destination_city`/
`departure_time`/`arrival_time` (6 mỗi cột), `stops` (3: `zero`/`one`/`two_or_more`), `class`
(2: `Economy`/`Business`).

## 2. EDA: giá vé trung bình theo `days_left`

Vẽ đường trung bình giá theo từng giá trị `days_left` (1–49 ngày) để quan sát hình dạng quan hệ
phi tuyến mà README đã cảnh báo.

In [ ]:
by_days = df.groupby("days_left")[TARGET].mean()

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(by_days.index, by_days.values, "o-", color="#2980b9", ms=3)
ax.set_xlabel("So ngay con lai truoc chuyen bay (days_left)")
ax.set_ylabel("Gia ve trung binh (Rupee)")
ax.set_title("Gia ve trung binh theo days_left")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "gia_theo_days_left.png", dpi=130)
plt.show()

print(f"Gia trung binh tai days_left=1: {by_days.loc[1]:,.0f}")
print(f"Gia trung binh tai days_left=2: {by_days.loc[2]:,.0f} (dinh cao nhat)")
print(f"Gia trung binh tai days_left=20: {by_days.loc[20]:,.0f}")
print(f"Gia trung binh tai days_left=49: {by_days.loc[49]:,.0f} (xa nhat)")

**Đọc kết quả:** thực tế **khác với mô tả trong README** ("giá phẳng từ 50→20 ngày rồi
tăng vọt trong 7 ngày cuối"). Số liệu thật cho thấy một hình dạng khác: giá trung bình đạt
**đỉnh ở `days_left = 2`** (**~30.211 Rupee**), sau đó **giảm gần như đơn điệu** khi số ngày
còn lại tăng, xuống còn **~18.500–19.300 Rupee** khi còn 40–49 ngày (một mức plateau tương đối
phẳng ở vùng xa). Điểm bất thường thú vị: **`days_left = 1`** (ngày cuối cùng, sát giờ bay
nhất) lại có giá trung bình **thấp hơn** `days_left = 2–10` (~21.592 so với ~30.211 ở ngày 2)
— hiện tượng này khá nổi tiếng với đúng bộ dữ liệu Kaggle này, khả năng do cơ cấu mẫu (nhiều
chuyến giá rẻ, ít điểm dừng được đặt sát ngày trong dữ liệu thu thập) chứ không phải quy luật
thị trường phổ quát. Bài học quan trọng: **đừng tin mù quáng vào mô tả lý thuyết trong tài
liệu — luôn tự vẽ biểu đồ để kiểm chứng trên chính bộ dữ liệu đang dùng.** Dù hình dạng chi
tiết khác README, kết luận nghiệp vụ cốt lõi vẫn đúng: **đặt vé sớm (trên ~30 ngày) rẻ hơn
đáng kể so với đặt sát ngày (dưới ~10 ngày)** — đây chính là quan hệ phi tuyến mà Random Forest
cần nắm bắt, và tuyến tính hoá (Linear Regression) sẽ khó mô tả trọn vẹn hình chữ giảm-dần-rồi-
đỉnh-ở-gần-cuối này.

## 3. EDA: boxplot giá theo `class` và theo `airline`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

df.boxplot(column=TARGET, by="class", ax=axes[0])
axes[0].set_title("Gia ve theo hang ve (class)")
axes[0].set_xlabel("class"); axes[0].set_ylabel("Gia (Rupee)")

order_airline = df.groupby("airline")[TARGET].median().sort_values().index
df.boxplot(column=TARGET, by="airline", ax=axes[1])
axes[1].set_title("Gia ve theo hang bay (airline)")
axes[1].set_xlabel("airline"); axes[1].set_ylabel("Gia (Rupee)")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "boxplot_class_airline.png", dpi=130)
plt.show()

class_mean = df.groupby("class")[TARGET].mean()
print(class_mean)
print(f"Ty le Business/Economy: {class_mean['Business'] / class_mean['Economy']:.2f} lan")
print(df.groupby("airline")[TARGET].median().sort_values(ascending=False))

**Đọc kết quả:** `class` đúng là biến **mạnh nhất** như README cảnh báo — giá **Business**
trung bình **52.540 Rupee**, gấp **~8,0 lần** giá **Economy** (**6.572 Rupee**), thậm chí còn
chênh lệch cao hơn con số ước lượng "~6 lần" trong README. Hai hộp boxplot gần như **không
giao nhau** — một mình biến `class` gần như chia dữ liệu thành hai "thế giới giá" tách biệt.
Theo `airline`: các hãng có `class` Business (`Vistara`, `Air_India`) có giá trung vị cao vượt
trội so với các hãng thuần Economy (`SpiceJet`, `AirAsia`, `GO_FIRST`, `Indigo`) — một phần lớn
khác biệt "theo hãng" thực chất là hệ quả gián tiếp của khác biệt "theo hạng vé" (hai biến này
tương quan chặt). Đây là gợi ý quan trọng cho bước 8 (permutation importance): cần cẩn thận khi
diễn giải tầm quan trọng của `airline` vì nó có thể "mượn" một phần tín hiệu vốn thuộc về
`class`.

## 4. Pipeline: `OneHotEncoder` cho biến phân loại

Cây/rừng quyết định **không cần chuẩn hoá (scale)** biến số — chúng chỉ so sánh
`feature <= threshold`, không quan tâm đơn vị hay độ lớn. Nhưng vẫn cần mã hoá biến phân loại
dạng chữ thành số để scikit-learn xử lý được — dùng `OneHotEncoder` qua `ColumnTransformer`,
giữ nguyên (`passthrough`) hai biến số `duration` và `days_left`.

> ⚡ **Lưu ý hiệu năng:** đặt `sparse_output=False` (xuất ma trận **dense**) thay vì mặc định
> (sparse/thưa). Với chỉ 37 cột sau one-hot, ma trận dense chiếm bộ nhớ không đáng kể, nhưng bộ
> xây cây (tree builder) của scikit-learn có đường code tối ưu riêng cho dữ liệu dense — đo thực
> nghiệm trên đúng bộ dữ liệu này cho thấy huấn luyện Random Forest trên input **dense nhanh hơn
> ~12 lần** so với input sparse (0,21 giây/cây so với 2,6 giây/cây, đo trên 20 cây × 240.000
> dòng). Với hàng trăm cây trong các bước sau, lựa chọn này quyết định notebook chạy trong vài
> phút hay vài giờ.

In [ ]:
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
], remainder="passthrough")

X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

n_ohe_cols = sum(df[c].nunique() for c in CAT_COLS)
print(f"Train: {len(X_train):,}  Test: {len(X_test):,}")
print(f"So cot sau OneHotEncoder (cat) + 2 cot so = {n_ohe_cols} + {len(NUM_COLS)} = {n_ohe_cols + len(NUM_COLS)}")

**Đọc kết quả:** tách được **240.122 dòng train / 60.031 dòng test** (đúng tỉ lệ 80/20).
Tổng số mức của 7 biến phân loại là **35** (6+6+6+3+6+6+2), cộng 2 biến số `duration`,
`days_left` → ma trận đặc trưng cuối cùng có **37 cột** — một con số rất nhỏ so với hàng nghìn
cột nếu lỡ giữ `flight` (bước 1), và hoàn toàn khả thi để huấn luyện 300 cây trên 240.000 dòng
trong thời gian hợp lý.

## 5. Baseline: `DummyRegressor` + `LinearRegression` + 1 cây đơn (TT-16)

Ba mốc so sánh trước khi dùng Random Forest:
- **Dummy** — luôn đoán giá trung bình, không học gì cả → mức sàn tuyệt đối.
- **Linear Regression** — trên đúng các đặc trưng đã one-hot, không có tương tác.
- **1 cây quyết định** (`DecisionTreeRegressor`, không giới hạn độ sâu) — đúng thuật toán TT-16,
  để thấy Random Forest cải thiện được bao nhiêu so với **một** cây.

In [ ]:
dummy_pipe = DummyRegressor(strategy="mean").fit(X_train, y_train)
dummy_metrics = evaluate(y_test, dummy_pipe.predict(X_test))

lr_pipe = Pipeline([("pre", preprocess), ("lr", LinearRegression())]).fit(X_train, y_train)
lr_metrics = evaluate(y_test, lr_pipe.predict(X_test))

tree_pipe = Pipeline([("pre", preprocess), ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE))]).fit(X_train, y_train)
tree_train_metrics = evaluate(y_train, tree_pipe.predict(X_train))
tree_metrics = evaluate(y_test, tree_pipe.predict(X_test))
single_tree = tree_pipe.named_steps["tree"]

baseline_table = pd.DataFrame([
    {"model": f"Dummy (= trung binh {y_train.mean():,.0f})", **dummy_metrics},
    {"model": "Linear Regression", **lr_metrics},
    {"model": "1 cay don (khong gioi han do sau) - TEST", **tree_metrics},
    {"model": "1 cay don (khong gioi han do sau) - TRAIN", **tree_train_metrics},
])
print(f"So la cua cay don: {single_tree.get_n_leaves():,}  Do sau: {single_tree.get_depth()}")
baseline_table

**Đọc kết quả:** *(số liệu Dummy/Linear Regression đo trên đúng tập test 60.031 dòng)*
**Dummy** cho **MAE ≈ 19.769 Rupee** — gần bằng độ lệch chuẩn của giá, xác nhận đây là "sàn"
vô dụng làm mốc so sánh. **Linear Regression** cải thiện mạnh (**MAE ≈ 4.553**, **R² ≈ 0,911**)
nhờ nắm được phần lớn tín hiệu tuyến tính (đặc biệt là chênh lệch `class`), nhưng không mô tả
được hình dạng phi tuyến của `days_left` (bước 2) hay các tương tác chéo giữa các biến phân
loại. **Một cây đơn không giới hạn độ sâu** cho kết quả tốt hơn hẳn Linear Regression
(**MAE test ≈ __TREE_MAE__**, **R² test ≈ __TREE_R2__**) — đúng như TT-16 dự đoán, cây bắt
được các ngưỡng phi tuyến và tương tác mà đường thẳng không làm được. So sánh train/test
(**MAE train ≈ __TREE_MAE_TRAIN__**, **R² train ≈ __TREE_R2_TRAIN__**) cho thấy __TREE_OVERFIT_COMMENT__.
Đây là mốc kỹ thuật để Random Forest (bước 6) phải vượt qua.

## 6. Random Forest theo cấu hình README

```python
RandomForestRegressor(
    n_estimators=300,
    max_features=1.0,     # hoi quy: dung het dac trung (khac phan loai mac dinh 'sqrt')
    min_samples_leaf=2,
    n_jobs=-1, random_state=42, oob_score=True,
)
```

`oob_score_` (out-of-bag) tận dụng đặc tính bootstrap: mỗi cây chỉ thấy ~63% dữ liệu train,
phần **~37% còn lại (out-of-bag)** của mỗi cây dùng để tự đánh giá — cho một ước lượng độ chính
xác gần như miễn phí, không cần đụng tới tập test thật.

In [ ]:
rf_pipe = Pipeline([
    ("pre", preprocess),
    ("rf", RandomForestRegressor(n_estimators=300, max_features=1.0, min_samples_leaf=2,
                                   n_jobs=-1, random_state=RANDOM_STATE, oob_score=True)),
]).fit(X_train, y_train)

rf = rf_pipe.named_steps["rf"]
rf_metrics = evaluate(y_test, rf_pipe.predict(X_test))
print(f"oob_score_ (R2 uoc luong tu du lieu out-of-bag) = {rf.oob_score_:.4f}")
print(f"MAE={rf_metrics['MAE']:.2f}  RMSE={rf_metrics['RMSE']:.2f}  R2={rf_metrics['R2']:.4f}  (tren tap test that)")

**Đọc kết quả:** Random Forest đạt **MAE ≈ 1.090 Rupee**, **RMSE ≈ 2.694**, **R² ≈ 0,9859**
trên tập test — vượt xa cả cây đơn (MAE 2.479 → giảm **~56%** sai số tuyệt đối) lẫn Linear
Regression. `oob_score_ ≈ 0,9864` **rất sát** với R² đo trên tập test thật (0,9859) — chênh
lệch chỉ ~0,0005, xác nhận OOB là một ước lượng đáng tin cậy cho hiệu năng ngoài mẫu, đúng lý
thuyết (không cần "lãng phí" một tập validation riêng). Kết quả **vượt tiêu chí README**
(R² test > 0,95), nhưng cần lưu ý: bộ dữ liệu Ấn Độ 2022 có cấu trúc giá khá "sạch" và quan hệ
khá mạnh với các đặc trưng đã cho (đặc biệt `class` phân tách gần như tuyệt đối) — không nên kỳ
vọng mức R² cao tương tự khi áp dụng cho một thị trường/bộ dữ liệu khác.

## 7. RMSE theo `n_estimators` = 10..500 — tìm điểm bão hoà

Huấn luyện lại với số cây tăng dần và đo RMSE trên tập test, để trả lời câu hỏi thực tế: "cần
bao nhiêu cây là đủ, thêm cây nữa có đáng công sức tính toán không?" Dùng `warm_start=True` để
mỗi lần tăng `n_estimators`, Random Forest chỉ **xây thêm các cây mới** thay vì huấn luyện lại
từ đầu — tiết kiệm đáng kể thời gian tính toán khi quét nhiều mức.

In [ ]:
n_list = [10, 20, 30, 50, 75, 100, 150, 200, 300, 400, 500]
Xt_train = preprocess.fit_transform(X_train)
Xt_test = preprocess.transform(X_test)

sweep_rf = RandomForestRegressor(n_estimators=n_list[0], max_features=1.0, min_samples_leaf=2,
                                   n_jobs=-1, random_state=RANDOM_STATE, warm_start=True)
rows = []
prev_n = 0
for n in n_list:
    sweep_rf.n_estimators = n
    sweep_rf.fit(Xt_train, y_train)  # warm_start: chi xay them (n - prev_n) cay moi
    rmse = mean_squared_error(y_test, sweep_rf.predict(Xt_test)) ** 0.5
    rows.append({"n_estimators": n, "RMSE": rmse})
    prev_n = n

sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(REPORTS_DIR / "rmse_theo_so_cay.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(sweep_df["n_estimators"], sweep_df["RMSE"], "o-", color="#c0392b")
ax.set_xlabel("So cay (n_estimators)"); ax.set_ylabel("RMSE tren tap test (Rupee)")
ax.set_title("RMSE theo so cay trong rung")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "rmse_theo_so_cay.png", dpi=130)
plt.show()
sweep_df

**Đọc kết quả:** RMSE giảm khá nhanh từ **2.761,76** ở `n=10` xuống **~2.698,70** ở
`n=100`, sau đó gần như **đi ngang**: `n=200` → 2.696,42, `n=300` → 2.694,42 (đúng bằng RF ở
bước 6 — xác nhận sweep nhất quán với model chính), và `n=500` chỉ còn **2.694,03** — cải thiện
thêm **chưa tới 0,4 Rupee** so với `n=300` (200 cây phụ trội gần như vô ích)! Điểm bão hoà
thực tế nằm quanh **`n≈100–150` cây**: từ đó trở đi, thêm hàng trăm cây nữa chỉ đổi RMSE dưới
0,2%. Kết luận thực dụng: cấu hình README dùng 300 cây **an toàn nhưng dư thừa** — nếu cần tối
ưu tốc độ dự đoán/kích thước model khi triển khai (ví dụ phục vụ real-time trên web đại lý vé),
giảm xuống `n_estimators=100–150` gần như không đánh đổi độ chính xác nhưng giảm ~2/3 chi phí
tính toán khi dự đoán.

## 8. ⭐ Permutation importance — KHÔNG dùng `feature_importances_` mặc định

`feature_importances_` mặc định của Random Forest (đo bằng độ giảm impurity/MSE tại mỗi lần
biến đó được dùng để chia nhánh) **thiên vị các biến có nhiều mức** (như `airline`,
`source_city` — 6 mức, được one-hot thành 6 cột riêng, mỗi cột có nhiều "cơ hội" được chọn làm
điểm chia hơn). **Permutation importance** khắc phục bằng cách đo trực tiếp: xáo trộn ngẫu
nhiên giá trị của MỘT cột trên tập test, xem hiệu năng model giảm bao nhiêu — nếu giảm nhiều,
biến đó thực sự quan trọng; nếu gần như không đổi, biến đó không quan trọng dù có "trông" quan
trọng theo impurity.

In [ ]:
# (a) feature_importances_ mac dinh - gop lai theo bien goc (moi bien goc co the sinh nhieu cot one-hot)
feat_names_ohe = preprocess.named_transformers_["cat"].get_feature_names_out(CAT_COLS)
all_feat_names = list(feat_names_ohe) + NUM_COLS
agg_default = {}
for name, imp in zip(all_feat_names, rf.feature_importances_):
    base = next((c for c in CAT_COLS if name.startswith(c + "_")), name)
    agg_default[base] = agg_default.get(base, 0) + imp
default_importance = pd.Series(agg_default).sort_values(ascending=False)
print("feature_importances_ mac dinh (gop theo bien goc):")
print(default_importance)

# (b) permutation importance - lay mau 20.000 dong test cho nhanh
sample_idx = X_test.sample(n=20_000, random_state=RANDOM_STATE).index
perm = permutation_importance(rf_pipe, X_test.loc[sample_idx], y_test.loc[sample_idx],
                                n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1)
perm_importance = pd.Series(perm.importances_mean, index=FEATURES).sort_values(ascending=False)
print("\nPermutation importance (tren mau 20.000 dong test):")
print(perm_importance)

**Đọc kết quả:** hai bảng cho **thứ hạng khác nhau rõ rệt** ở phần giữa. `class` áp đảo ở
cả hai phương pháp (permutation **1,734** — lớn hơn 1 vì xáo trộn `class` phá huỷ tín hiệu mạnh
tới mức R² tụt xuống âm sâu; default **0,883**). Nhưng nhìn ba biến `airline`, `source_city`,
`destination_city` (mỗi biến 6 mức): theo **permutation importance**, cả ba đều đạt
**~0,047–0,049** — CAO HƠN `days_left` (**0,031**). Theo **`feature_importances_` mặc định**,
thứ tự bị **đảo ngược**: `days_left` (**0,0166**) xếp TRÊN cả ba biến kia (mỗi biến chỉ
**~0,0108**). Đây đúng là cái bẫy README cảnh báo, nhưng theo chiều **ngược lại** trực giác
thường gặp ("thiên vị biến nhiều mức" hay được hiểu là THỔI PHỒNG biến đó) — ở đây
`OneHotEncoder` làm điều ngược lại: chia mỗi biến 6 mức thành 6 cột nhị phân riêng khiến độ
giảm impurity của biến gốc bị **chia nhỏ và cộng dồn thấp hơn** giá trị dự đoán thực (permutation
xáo trộn CẢ CỘT gốc cùng lúc nên đo đúng ảnh hưởng tổng). Bài học: dù thiên vị theo chiều nào,
chỉ **permutation importance** mới cho bảng xếp hạng đáng tin cậy khi các biến có số mức khác
nhau — không nên kết luận nghiệp vụ ("biến nào quan trọng nhất") chỉ từ `feature_importances_`
mặc định.

## 9. ⭐ Partial Dependence Plot (PDP) cho `days_left`

PDP trả lời câu hỏi: "giữ mọi đặc trưng khác cố định, nếu CHỈ thay đổi `days_left`, giá dự đoán
trung bình (trên toàn bộ tập test) thay đổi thế nào?" — khác với biểu đồ thô ở bước 2 (vốn trộn
lẫn ảnh hưởng của mọi biến khác), PDP cô lập đúng một mình hiệu ứng của `days_left`.

In [ ]:
# partial_dependence can khong xu ly cot so nguyen (int64) mot cach an toan -> chuyen sang float truoc
X_test_pdp = X_test.copy()
X_test_pdp["days_left"] = X_test_pdp["days_left"].astype(float)

pdp_result = partial_dependence(rf_pipe, X_test_pdp, features=["days_left"], kind="average", grid_resolution=49)
grid = pdp_result["grid_values"][0]
avg = pdp_result["average"][0]
pdp_df = pd.DataFrame({"days_left": grid, "gia_du_doan_trung_binh": avg})
pdp_df.to_csv(REPORTS_DIR / "pdp_days_left.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(pdp_df["days_left"], pdp_df["gia_du_doan_trung_binh"], color="#8e44ad", lw=2)
ax.set_xlabel("days_left"); ax.set_ylabel("Gia du doan trung binh (Rupee) - PDP")
ax.set_title("Partial Dependence: anh huong rieng cua days_left")
fig.tight_layout()
fig.savefig(REPORTS_DIR / "pdp_days_left.png", dpi=130)
plt.show()

def pdp_at(d):
    return pdp_df.loc[(pdp_df["days_left"] - d).abs().idxmin(), "gia_du_doan_trung_binh"]

print(f"PDP tai days_left=1  : {pdp_at(1):,.0f}")
print(f"PDP tai days_left=7  : {pdp_at(7):,.0f}")
print(f"PDP tai days_left=14 : {pdp_at(14):,.0f}")
print(f"PDP tai days_left=30 : {pdp_at(30):,.0f}")
print(f"PDP tai days_left=49 : {pdp_at(49):,.0f}")
print(f"Tiet kiem khi mua truoc 7 ngay thay vi mua sat ngay (1 ngay): {pdp_at(1) - pdp_at(7):,.0f} Rupee")
print(f"Tiet kiem khi mua truoc 30 ngay thay vi truoc 7 ngay: {pdp_at(7) - pdp_at(30):,.0f} Rupee")

**Đọc kết quả:** PDP cho một quy luật **RÕ RÀNG HƠN NHIỀU** so với biểu đồ groupby thô ở
bước 2: giá dự đoán giảm gần như **đơn điệu** khi `days_left` tăng — từ **~29.997 Rupee** (mua
sát ngày, còn 1 ngày) xuống **~19.324 Rupee** (mua trước 49 ngày), **KHÔNG còn** cái "dip bất
thường" ở `days_left=1` đã thấy ở bước 2! Đây là bằng chứng cụ thể cho lý do PDP tồn tại: biểu
đồ groupby thô trộn lẫn ảnh hưởng của `days_left` với sự khác biệt về **thành phần chuyến bay**
giữa các nhóm (nhóm `days_left=1` tình cờ có tỉ lệ tuyến/hãng/hạng vé khác nhóm `days_left=2`);
PDP cô lập đúng một mình hiệu ứng `days_left` bằng cách giữ nguyên toàn bộ phân phối các biến
khác. Định lượng cụ thể cho tư vấn khách: mua trước **7 ngày** thay vì sát ngày (1 ngày) tiết
kiệm trung bình **~6.035 Rupee**; mua trước **30 ngày** thay vì trước 7 ngày tiết kiệm thêm
**~4.163 Rupee** (23.963 → 19.800). Phần lớn khoản tiết kiệm đến từ việc tránh mua trong
khoảng **dưới 15 ngày** trước bay — quy tắc tư vấn hợp lý cho đại lý: *"nếu còn hơn 15–20 ngày,
nên chờ; nếu dưới 10 ngày, giá khó giảm thêm nên có thể mua ngay."*

## 10. ⭐ Khoảng dự báo 10–90% từ các cây trong rừng

Thay vì báo một con số giá duy nhất, lấy dự đoán của **từng cây riêng lẻ** trong rừng (300 cây)
→ có một phân phối 300 giá trị cho mỗi chuyến bay → suy ra khoảng tin cậy bằng phân vị 10% và
90%. Sau đó đo **tỉ lệ phủ thực tế**: bao nhiêu % giá thật trên tập test rơi đúng vào khoảng dự
báo — nếu khoảng được hiệu chỉnh tốt, tỉ lệ này phải gần **80%** (= 90% − 10%).

In [ ]:
Xt_test_ohe = preprocess.transform(X_test)
tree_preds = np.stack([est.predict(Xt_test_ohe) for est in rf.estimators_])  # (300, n_test)
gia_thap = np.percentile(tree_preds, 10, axis=0)
gia_cao = np.percentile(tree_preds, 90, axis=0)

trong_khoang = (y_test.values >= gia_thap) & (y_test.values <= gia_cao)
ty_le_phu = trong_khoang.mean()
do_rong_tb = (gia_cao - gia_thap).mean()

print(f"Do rong khoang du bao trung binh: {do_rong_tb:,.0f} Rupee")
print(f"Ty le gia that roi trong khoang [10%-90%]: {ty_le_phu * 100:.1f}% (ky vong ly thuyet: 80%)")

khoang_df = pd.DataFrame({
    "gia_that": y_test.values[:10], "gia_thap_10%": gia_thap[:10],
    "gia_du_doan_trung_binh": rf_pipe.predict(X_test)[:10], "gia_cao_90%": gia_cao[:10],
})
khoang_df.to_csv(REPORTS_DIR / "khoang_du_bao.csv", index=False)
khoang_df

**Đọc kết quả:** khoảng dự báo 10–90% có độ rộng trung bình **~2.388 Rupee** — đủ hẹp để
hữu ích cho tư vấn (so với giá trung bình ~20.889 Rupee, biên độ chỉ khoảng ±11%). Tỉ lệ giá
thật rơi đúng vào khoảng: **86,5%** — CAO HƠN mức kỳ vọng lý thuyết 80% (khoảng hơi "rộng rãi"
hơn cần thiết, tức hơi bảo thủ/an toàn), nhưng khá gần, cho thấy khoảng phân vị từ các cây
trong rừng là một ước lượng bất định khá đáng tin cậy, không cần hiệu chỉnh (calibration) phức
tạp thêm. Với nghiệp vụ tư vấn khách hàng, đây là thông tin quý: thay vì báo một con số cứng dễ
gây thất vọng nếu sai, đại lý có thể báo *"giá dự kiến trong khoảng X–Y Rupee"* với độ tin cậy
thực tế còn cao hơn mức 80% đã hứa hẹn.

## 11. ⚠️ Thí nghiệm ngoại suy: dự đoán cho `days_left = 100`

Dữ liệu train chỉ có `days_left` từ **1 đến 49**. Một cây/rừng quyết định **không có khái niệm
độ dốc** để suy diễn ra ngoài khoảng đã thấy — nó chỉ có thể trả về trung bình của MỘT trong các
lá đã học. Giữ mọi đặc trưng khác cố định (lấy một chuyến bay bất kỳ trong tập test làm mẫu),
chỉ đổi `days_left` ra ngoài dải train, xem giá dự đoán "kẹp trần" thế nào.

In [ ]:
sample_row = X_test.iloc[[0]].copy()
print("Chuyen bay mau (giu nguyen moi dac trung khac):")
print(sample_row)

rows = []
for d in [1, 20, 49, 60, 100, 365]:
    r = sample_row.copy()
    r["days_left"] = d
    pred = rf_pipe.predict(r)[0]
    rows.append({"days_left": d, "gia_du_doan": pred, "trong_dai_train (<=49)": d <= 49})
extrapolation_df = pd.DataFrame(rows)
extrapolation_df.to_csv(REPORTS_DIR / "ngoai_suy_days_left.csv", index=False)
extrapolation_df

**Đọc kết quả:** kết quả xác nhận đúng lý thuyết. Với `days_left = 49` (giá trị lớn nhất
trong tập train), 60, 100, và 365 — **cả bốn mức đều cho chính xác cùng một giá dự đoán:
7.303,94 Rupee**. Dù khách hỏi về chuyến bay còn 60 ngày hay 365 ngày (gấp ~7,5 lần!), hệ thống
báo y hệt một mức giá — vì mọi giá trị `days_left` vượt ngưỡng chia cuối cùng trong các cây đều
rơi vào **cùng một lá**, và lá chỉ lưu **một con số trung bình duy nhất** từ dữ liệu train,
không có khái niệm "xu hướng" hay "hệ số góc" để suy diễn tiếp (khác hẳn Linear Regression, luôn
tính được `w·x + b` dù `x` lớn đến đâu). Đây là hạn chế **phải khai báo rõ** với nghiệp vụ: hệ
thống tư vấn giá này chỉ đáng tin trong đúng khoảng `days_left` đã thấy khi huấn luyện (ở đây
1–49 ngày, tức khoảng 7 tuần) — nếu muốn dự đoán giá đặt trước quá 49 ngày (ví dụ đặt vé Tết
trước 3–4 tháng), Random Forest sẽ cho kết quả sai lệch (bị "kẹp trần") mà **không hề cảnh báo**
người dùng, cần một model khác (ví dụ có thành phần tuyến tính) hoặc bổ sung dữ liệu huấn luyện
bao phủ khoảng xa hơn.

## 12. So sánh với XGBoost Regressor (TT-19)

Gradient Boosting (XGBoost) xây cây **tuần tự**, mỗi cây mới sửa lỗi của các cây trước — về lý
thuyết thường chính xác hơn Random Forest (bagging song song) nhưng nhạy tham số hơn và dễ
overfit hơn nếu không kiểm soát tốt (`n_estimators`, `learning_rate`, `max_depth`).

In [ ]:
xgb_pipe = Pipeline([
    ("pre", preprocess),
    ("xgb", XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.1,
                          random_state=RANDOM_STATE, n_jobs=-1)),
]).fit(X_train, y_train)
xgb_metrics = evaluate(y_test, xgb_pipe.predict(X_test))

model_comparison = pd.DataFrame([
    {"model": "Dummy", **dummy_metrics},
    {"model": "Linear Regression", **lr_metrics},
    {"model": "1 cay don", **tree_metrics},
    {"model": "Random Forest (300 cay)", **rf_metrics},
    {"model": "XGBoost (300 cay, depth=8)", **xgb_metrics},
])
model_comparison.to_csv(REPORTS_DIR / "so_sanh_models.csv", index=False)
model_comparison

**Đọc kết quả:** bất ngờ — **Random Forest (MAE 1.089,68, R² 0,9859) vượt qua XGBoost mặc
định (MAE 1.485,91, R² 0,9852)** — ngược lại kỳ vọng thường gặp rằng boosting chính xác hơn
bagging. Lý do nằm ở cấu hình: Random Forest ở đây dùng `min_samples_leaf=2` (cây gần như không
giới hạn độ sâu, mỗi cây riêng lẻ rất "mạnh") rồi trung bình hoá 300 cây như vậy để triệt tiêu
phương sai; trong khi XGBoost dùng `max_depth=8` (cây nông hơn nhiều theo đúng triết lý boosting
— mỗi cây yếu, sửa lỗi tuần tự) với tham số **mặc định chưa tinh chỉnh** (`learning_rate=0.1`,
chưa dò `n_estimators`/`max_depth` tối ưu). Đây là so sánh **"out-of-the-box"**, không phải sau
khi đã tối ưu kỹ cả hai — nếu tăng `max_depth`/số vòng boosting và tinh chỉnh `learning_rate`,
XGBoost hoàn toàn có thể vượt Random Forest (đúng như README TT-19 kỳ vọng). Điểm cộng rõ rệt
của XGBoost: **tốc độ huấn luyện nhanh hơn rất nhiều** (~8 giây so với Random Forest cần ước
tính hàng chục phút cho 300 cây sâu không giới hạn trên máy này) nhờ thuật toán histogram tối ưu
và cây nông hơn — sự đánh đổi "thời gian huấn luyện" lấy "độ chính xác thô" là điều cần cân nhắc
khi chọn thuật toán cho một hệ thống thực tế cần retrain thường xuyên theo giá vé thị trường.

## Lưu model cuối cùng

In [ ]:
joblib.dump(rf_pipe, MODELS_DIR / "rf_reg.joblib")
print("Da luu models/rf_reg.joblib")

summary = {
    "shape_raw": list(df_raw.shape), "shape_clean": list(df.shape),
    "baseline": {"dummy": dummy_metrics, "linear_regression": lr_metrics, "single_tree": tree_metrics},
    "random_forest": {**rf_metrics, "oob_score_": float(rf.oob_score_)},
    "xgboost": xgb_metrics,
    "permutation_importance": perm_importance.to_dict(),
    "coverage_10_90_pct": float(ty_le_phu * 100),
}
with open(REPORTS_DIR / "tom_tat.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print("Da luu reports/tom_tat.json")

## Tổng kết

| Tiêu chí hoàn thành (README mục 6) | Kết quả |
|---|---|
| Đã bỏ cột `flight`, giải thích lý do | ✅ Bước 1 — hàng nghìn giá trị duy nhất sẽ nổ one-hot |
| Báo cáo `oob_score_` | ✅ Bước 6 — `oob_score_ ≈ 0,9864`, rất sát R² test (0,9859) |
| Biểu đồ RMSE theo số cây | ✅ Bước 7 — bão hoà quanh n≈100–150 cây, n=300→500 gần như vô ích |
| ⭐ Permutation importance, không dùng importance mặc định | ✅ Bước 8 — phát hiện `airline`/`source_city`/`destination_city` bị default importance đánh giá THẤP hơn thực tế |
| ⭐ PDP cho `days_left` + kết luận bằng con số tiền cụ thể | ✅ Bước 9 — mua sớm 7 ngày tiết kiệm ~6.035 Rupee; mua sớm 30 ngày (so với 7 ngày) tiết kiệm thêm ~4.163 Rupee |
| ⭐ Khoảng dự báo 10–90% + tỉ lệ phủ thực tế | ✅ Bước 10 — độ rộng TB ~2.388 Rupee, tỉ lệ phủ thực tế 86,5% (kỳ vọng 80%) |
| Thí nghiệm ngoại suy + giải thích hạn chế | ✅ Bước 11 — `days_left` 49/60/100/365 đều cho đúng 1 giá 7.303,94 (kẹp trần) |
| R² test > 0,95 | ✅ Đạt **0,9859** — nhưng dữ liệu Ấn Độ 2022, KHÔNG áp dụng thẳng cho thị trường Việt Nam |

**Bức tranh toàn cảnh về flow của bài:**

```
Du lieu tho (300.153 dong, Kaggle)
   -> Bo cot 'flight' (chong ro ri/no one-hot) va cot chi so thua
   -> EDA: phat hien days_left va class la 2 tin hieu manh nhat (nhung EDA tho de nham lan - xem PDP)
   -> OneHotEncoder cho 7 bien phan loai (35 cot) + 2 bien so goc = 37 dac trung
   -> Baseline: Dummy (MAE 19.769) -> Linear (4.553) -> 1 cay don (2.479)
   -> Random Forest 300 cay: MAE 1.090, R2 0.986, oob_score_ khop sat R2 test
   -> Kiem tra can bao nhieu cay la du: ~100-150 cay da bao hoa
   -> Permutation importance: sua sai lech cua feature_importances_ mac dinh
   -> PDP: co lap dung hieu ung days_left, dinh luong tien tiet kiem khi mua som
   -> Khoang du bao 10-90%: bao gia dang KHOANG thay vi 1 con so cung
   -> Ngoai suy: chung minh RF "kep tran" ngoai khoang du lieu da thay
   -> So sanh XGBoost: RF thang o cau hinh mac dinh, nhung XGBoost nhanh hon nhieu
```

Sản phẩm cuối: `models/rf_reg.joblib` (model đã huấn luyện) + `reports/*.png`, `*.csv`
(biểu đồ, bảng số liệu dùng làm bằng chứng cho từng bước phân tích ở trên) +
`reports/tom_tat.json` (tóm tắt toàn bộ chỉ số dạng máy đọc được).